# Module 9: Final Campus-Event Model

Implement the M8 blueprint. Validation chooses the model; the final test set is used once.

<table width="100%"><tr><td>&#128269;&nbsp;<b>Stuck on this code?</b></td><td align="right"><a href="VIDEO_LINK_HERE"><img src="https://img.shields.io/badge/Watch_Video-red?style=flat&logo=youtube&logoColor=white"></a></td></tr></table>

In [ ]:
# Load the cleaned dataset and the "blueprint" (the plan) made back in Module 8.
import json
from pathlib import Path
import pandas as pd

df=pd.read_csv('campus_events_clean.csv')
# Use the final blueprint if it exists, otherwise fall back to the checkpoint version.
bp_path=Path('m8_model_blueprint.json') if Path('m8_model_blueprint.json').exists() else Path('m8_model_blueprint_checkpoint.json')
blueprint=json.loads(bp_path.read_text())
print('Blueprint:',bp_path.name,'Dataset:',df.shape)

## 1. Confirm the M8 blueprint
Record any revision before modeling. Do not quietly add rejected or post-event features.

<table width="100%"><tr><td>&#128269;&nbsp;<b>Stuck on this code?</b></td><td align="right"><a href="VIDEO_LINK_HERE"><img src="https://img.shields.io/badge/Watch_Video-red?style=flat&logo=youtube&logoColor=white"></a></td></tr></table>

In [ ]:
target=blueprint['target']
features=blueprint['accepted_features']
# These columns would "cheat" (they're only known after the event happens), so they must never be used as inputs.
blocked={'event_id','actual_weather','actual_attendance','attendance_rate','high_turnout'}
assert not blocked.intersection(features), f'Blocked feature selected: {blocked.intersection(features)}'
print('Target:',target,'Features:',features)

## 2. Build a preprocessing pipeline
Numeric columns pass through; categorical columns are one-hot encoded. Fit preprocessing only on training data.

<table width="100%"><tr><td>&#128269;&nbsp;<b>Stuck on this code?</b></td><td align="right"><a href="VIDEO_LINK_HERE"><img src="https://img.shields.io/badge/Watch_Video-red?style=flat&logo=youtube&logoColor=white"></a></td></tr></table>

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix, classification_report
# X = the input columns (features), y = the answer we want to predict (turned into 0/1 instead of no/yes).
X=df[features].copy(); y=df[target].map({'no':0,'yes':1})
# Split the data twice: first carve out a "test" set we won't touch until the very end,
# then split the rest into "train" (to fit the model) and "validation" (to compare models).
X_build,X_test,y_build,y_test=train_test_split(X,y,test_size=.20,random_state=42,stratify=y)
X_train,X_val,y_train,y_val=train_test_split(X_build,y_build,test_size=.25,random_state=42,stratify=y_build)
# Separate numeric columns (like counts) from categorical columns (like text labels), since they need different handling.
num=X.select_dtypes(include='number').columns.tolist(); cat=[c for c in X.columns if c not in num]
# For numbers: fill in missing values with the median, then scale them.
# For categories: fill in missing values with the most common value, then one-hot encode (turn labels into 0/1 columns).
pre=ColumnTransformer([('num',Pipeline([('imp',SimpleImputer(strategy='median')),('scale',StandardScaler())]),num),('cat',Pipeline([('imp',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore'))]),cat)])
print(len(X_train),len(X_val),len(X_test))

## 3. Establish the baseline

<table width="100%"><tr><td>&#128269;&nbsp;<b>Stuck on this code?</b></td><td align="right"><a href="VIDEO_LINK_HERE"><img src="https://img.shields.io/badge/Watch_Video-red?style=flat&logo=youtube&logoColor=white"></a></td></tr></table>

In [ ]:
# Baseline = "just guess the most common answer every time." Any real model should beat this.
baseline=int(y_train.mode().iloc[0]); baseline_val=float((y_val==baseline).mean()); print('Validation baseline:',round(baseline_val,3))

## 4. Compare candidates on validation data only

<table width="100%"><tr><td>&#128269;&nbsp;<b>Stuck on this code?</b></td><td align="right"><a href="VIDEO_LINK_HERE"><img src="https://img.shields.io/badge/Watch_Video-red?style=flat&logo=youtube&logoColor=white"></a></td></tr></table>

In [ ]:
# Try two different model types and see which one predicts better.
candidates={'decision_tree':DecisionTreeClassifier(max_depth=4,min_samples_leaf=8,random_state=42),'logistic_regression':LogisticRegression(max_iter=1000)}
validation=[]; fitted={}
for name,model in candidates.items():
 # Combine the preprocessing steps with the model, train on the training set, then predict on the validation set.
 pipe=Pipeline([('preprocess',pre),('model',model)]); pipe.fit(X_train,y_train); pred=pipe.predict(X_val); fitted[name]=pipe
 # Score each model: accuracy (overall correctness), precision (when it says "yes", how often is it right), recall (of all real "yes"es, how many did it catch).
 validation.append({'model':name,'accuracy':accuracy_score(y_val,pred),'precision':precision_score(y_val,pred,zero_division=0),'recall':recall_score(y_val,pred,zero_division=0)})
validation_table=pd.DataFrame(validation); validation_table

## 5. Lock the model, then refit on train + validation

<table width="100%"><tr><td>&#128269;&nbsp;<b>Stuck on this code?</b></td><td align="right"><a href="VIDEO_LINK_HERE"><img src="https://img.shields.io/badge/Watch_Video-red?style=flat&logo=youtube&logoColor=white"></a></td></tr></table>

In [ ]:
# Pick the model with the best accuracy (recall breaks ties), then retrain it on train+validation combined for the strongest final version.
chosen_name=validation_table.sort_values(['accuracy','recall'],ascending=False).iloc[0]['model']; chosen=Pipeline([('preprocess',pre),('model',candidates[chosen_name])]); chosen.fit(X_build,y_build); print('Locked:',chosen_name)

## 6. Use the final test set once

<table width="100%"><tr><td>&#128269;&nbsp;<b>Stuck on this code?</b></td><td align="right"><a href="VIDEO_LINK_HERE"><img src="https://img.shields.io/badge/Watch_Video-red?style=flat&logo=youtube&logoColor=white"></a></td></tr></table>

In [ ]:
# This is the ONE and ONLY time we use the test set, so these numbers reflect true, unbiased performance.
test_pred=chosen.predict(X_test); metrics={'accuracy':float(accuracy_score(y_test,test_pred)),'precision':float(precision_score(y_test,test_pred,zero_division=0)),'recall':float(recall_score(y_test,test_pred,zero_division=0))}; cm=confusion_matrix(y_test,test_pred,labels=[0,1]); print(metrics); print(pd.DataFrame(cm,index=['actual low','actual high'],columns=['predicted low','predicted high']))

## 7. Inspect mistakes and compare with the baseline

<table width="100%"><tr><td>&#128269;&nbsp;<b>Stuck on this code?</b></td><td align="right"><a href="VIDEO_LINK_HERE"><img src="https://img.shields.io/badge/Watch_Video-red?style=flat&logo=youtube&logoColor=white"></a></td></tr></table>

In [ ]:
test_results=X_test.copy(); test_results['actual']=y_test; test_results['predicted']=test_pred
# Grab one example of each mistake type: a false positive (predicted "yes" but actually "no")...
false_positive=test_results[(test_results.actual==0)&(test_results.predicted==1)].head(1)
# ...and a false negative (predicted "no" but actually "yes").
false_negative=test_results[(test_results.actual==1)&(test_results.predicted==0)].head(1)
print('Final majority baseline:',round(float((y_test==int(y_build.mode().iloc[0])).mean()),3)); display(false_positive); display(false_negative)

## 8. Stress-test the model edges
Use three fictional cases: unfamiliar category, extreme promotion, and changed forecast. These tests reveal sensitivity; they do not prove cause.

<table width="100%"><tr><td>&#128269;&nbsp;<b>Stuck on this code?</b></td><td align="right"><a href="VIDEO_LINK_HERE"><img src="https://img.shields.io/badge/Watch_Video-red?style=flat&logo=youtube&logoColor=white"></a></td></tr></table>

In [ ]:
# Take one real example and tweak it into three made-up edge cases, to see how the model reacts to unusual inputs.
stress=X_test.head(1).copy(); stress_cases=[]
for label,change in [('unfamiliar event type',{'event_type':'new_event_type'}),('extreme promotion',{'social_posts':999}),('changed forecast',{'weather_forecast':'storm'} )]:
 case=stress.copy()
 for k,v in change.items():
  if k in case.columns: case[k]=v
 stress_cases.append({'case':label,'prediction':int(chosen.predict(case)[0])})
pd.DataFrame(stress_cases)

## 9. Propose one improvement and record AI use

<table width="100%"><tr><td>&#128269;&nbsp;<b>Stuck on this code?</b></td><td align="right"><a href="VIDEO_LINK_HERE"><img src="https://img.shields.io/badge/Watch_Video-red?style=flat&logo=youtube&logoColor=white"></a></td></tr></table>

In [ ]:
# A short note on what could make the model better next time, plus a record of how AI help was used (for academic honesty).
improvement='Collect more examples for weak or unfamiliar event categories, then re-run validation before touching a new final test set.'
ai_use_record={'allowed':'AI explained code or suggested tests','student_owned':'feature decisions, metric interpretation, error analysis, limitations, final model card'}

## 10. Export the model card

<table width="100%"><tr><td>&#128269;&nbsp;<b>Stuck on this code?</b></td><td align="right"><a href="VIDEO_LINK_HERE"><img src="https://img.shields.io/badge/Watch_Video-red?style=flat&logo=youtube&logoColor=white"></a></td></tr></table>

In [ ]:
# Bundle everything about this project (results, limits, decisions) into one summary dict, then save it as a JSON file.
model_card={'project':'Will They Show Up?','purpose':'Low-stakes classroom prediction of fictional campus-event turnout','blueprint_used':bp_path.name,'features':features,'target':target,'selection_evidence':validation,'selected_model':chosen_name,'final_test_metrics':metrics,'confusion_matrix':cm.tolist(),'baseline':blueprint.get('majority_baseline_accuracy'),'known_limits':blueprint.get('predicted_failure_cases',[]),'stress_tests':stress_cases,'improvement':improvement,'prohibited_uses':blueprint.get('prohibited_uses',[]),'human_review_boundary':blueprint.get('human_review_boundary'),'ai_use_record':ai_use_record}
Path('m9_model_card.json').write_text(json.dumps(model_card,indent=2)); print('Saved m9_model_card.json')
model_card